# 02. Моделирование новых банковских продуктов

Задача формулируется как multi-label ranking: по профилю и текущему портфелю клиента в месяце $t$ нужно ранжировать 24 продукта, которые он **впервые подключит** в соседнем месяце $t+1$. Основная offline-метрика — MAP@7; дополнительно оцениваются Recall@7, Precision@7, catalog coverage@7 и macro PR-AUC.

Ноутбук загружает полный CSV одним вызовом `pandas.read_csv` и использует всех клиентов. Это требует значительного объёма RAM. Он не сохраняет и не перезаписывает production-модель: refit, экспорт `ml_models/model.joblib`, отчёты и MLflow выполняются штатным CLI, приведённым в конце. Outputs очищены после смены режима загрузки, поэтому ноутбук нужно выполнить заново.

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    candidate_root = PROJECT_ROOT.parent
    if (candidate_root / "src").is_dir():
        PROJECT_ROOT = candidate_root
    else:
        raise RuntimeError("Запустите ноутбук из корня проекта или каталога notebooks")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from bank_recommender.constants import (
    DATE_COLUMN,
    ID_COLUMN,
    PRODUCT_COLUMNS,
    PRODUCT_DESCRIPTIONS_RU,
)
from bank_recommender.data import build_temporal_pairs, read_snapshots
from bank_recommender.features import engineer_features
from bank_recommender.metrics import evaluate_ranking
from bank_recommender.model import (
    BankProductRecommender,
    PopularityRecommender,
    ranked_recommendations,
)

SEED = 42
TOP_K = 7
VALIDATION_PERIOD = "2016-04"  # признаки апреля -> покупки мая
random.seed(SEED)
np.random.seed(SEED)

raw_data_path = Path(os.getenv("DATA_PATH", "train_ver2.csv")).expanduser()
DATA_PATH = raw_data_path if raw_data_path.is_absolute() else PROJECT_ROOT / raw_data_path
MODEL_MAX_ITER = int(os.getenv("MODEL_MAX_ITER", "12"))
if MODEL_MAX_ITER < 1:
    raise ValueError("MODEL_MAX_ITER должен быть положительным")
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Датасет не найден: {DATA_PATH}")

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda value: f"{value:,.5f}")

print(f"Данные: {DATA_PATH}")
print(
    "Режим загрузки: все клиенты; "
    f"seed={SEED}; max_iter={MODEL_MAX_ITER}; holdout={VALIDATION_PERIOD}->2016-05"
)

## 1. Target и защита от утечки

Для клиента $u$ и продукта $p$ целевая переменная имеет вид

$$y_{u,p,t+1}=\max(product_{u,p,t+1}-product_{u,p,t}, 0).$$

То есть `0→1` — положительный класс, `0→0`, `1→1` и снятие `1→0` — ноль. Пара создаётся только при наличии соседних календарных месяцев. В признаки входят профиль и портфель из $t$, но не значения из $t+1$.

Случайный train/test split здесь некорректен: снимки одного клиента и более поздние месяцы проникли бы в обучение. Поэтому train содержит feature-месяцы до апреля 2016 года, а единственный holdout использует апрельские признаки для прогноза майских покупок. Все imputer, scaler и one-hot encoder обучаются внутри pipeline только на train.

In [ ]:
snapshots = read_snapshots(DATA_PATH)
pairs = build_temporal_pairs(snapshots)
x_train, y_train, x_valid, y_valid = pairs.split(VALIDATION_PERIOD)

valid_periods = x_valid[DATE_COLUMN].dt.to_period("M")
if not valid_periods.eq(pd.Period(VALIDATION_PERIOD, freq="M")).all():
    raise AssertionError("Holdout содержит строки вне апреля 2016 года")

split_summary = pd.DataFrame(
    {
        "train": [
            len(x_train),
            x_train[ID_COLUMN].nunique(),
            str(x_train[DATE_COLUMN].min().to_period("M")),
            str(x_train[DATE_COLUMN].max().to_period("M")),
            int(y_train[PRODUCT_COLUMNS].to_numpy().sum()),
            float((y_train[PRODUCT_COLUMNS].sum(axis=1) > 0).mean()),
        ],
        "validation (апрель→май)": [
            len(x_valid),
            x_valid[ID_COLUMN].nunique(),
            str(x_valid[DATE_COLUMN].min().to_period("M")),
            str(x_valid[DATE_COLUMN].max().to_period("M")),
            int(y_valid[PRODUCT_COLUMNS].to_numpy().sum()),
            float((y_valid[PRODUCT_COLUMNS].sum(axis=1) > 0).mean()),
        ],
    },
    index=[
        "пар",
        "клиентов",
        "минимальный feature-месяц",
        "максимальный feature-месяц",
        "событий 0→1",
        "доля клиентов с покупкой",
    ],
)
display(split_summary)

## 2. Дисбаланс target

Положительные события редки и неодинаково распределены по продуктам. Таблица помогает проверить, какие метки представлены в полном train и временном holdout. Нулевая частота отдельного продукта означает, что для него в соответствующем периоде не наблюдалось новых подключений; продукт при этом остаётся в каталоге.

In [ ]:
target_prevalence = pd.DataFrame(
    {
        "product": PRODUCT_COLUMNS,
        "description": [PRODUCT_DESCRIPTIONS_RU[p] for p in PRODUCT_COLUMNS],
        "train_events": [int(y_train[p].sum()) for p in PRODUCT_COLUMNS],
        "train_rate": [float(y_train[p].mean()) for p in PRODUCT_COLUMNS],
        "validation_events": [int(y_valid[p].sum()) for p in PRODUCT_COLUMNS],
        "validation_rate": [float(y_valid[p].mean()) for p in PRODUCT_COLUMNS],
    }
).sort_values("train_events", ascending=False, ignore_index=True)
display(target_prevalence.head(10))
display(target_prevalence.tail(5))

## 3. Baseline: глобальная популярность новых покупок

`PopularityRecommender` оценивает сглаженную частоту каждого события `0→1` на train и выдаёт один глобальный порядок продуктов. Это обязательная точка отсчёта: сложная модель имеет смысл только при улучшении временного holdout. Во время расчёта top-k уже имеющиеся у клиента продукты маскируются.

In [ ]:
baseline = PopularityRecommender().fit(x_train, y_train)
baseline_scores = baseline.predict_scores(x_valid)
comparison = {
    "popularity": evaluate_ranking(
        y_valid, baseline_scores, owned=x_valid, k=TOP_K
    )
}
display(pd.DataFrame(comparison).T)

## 4. Leakage-safe ML-pipeline

Обучаемая модель — One-vs-Rest набор линейных `SGDClassifier(loss='log_loss')`. Перед ним выполняются детерминированная генерация календарных и портфельных признаков, медианное заполнение и масштабирование чисел, заполнение пропусков и one-hot кодирование категорий. Неизвестные категории допускаются на inference.

Для быстрого запуска ноутбука используется небольшое `max_iter`. Это исследовательский прогон, а не production refit; предупреждения о недостижении сходимости подавляются только в этой демонстрационной ячейке.

In [ ]:
candidate = BankProductRecommender(
    alpha=0.0001,
    max_iter=MODEL_MAX_ITER,
    prior_weight=0.0,
    random_seed=SEED,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", message="Label .*", category=UserWarning)
    candidate.fit(x_train, y_train)

raw_model_scores = candidate.predict_scores(x_valid)
if raw_model_scores.shape != (len(x_valid), len(PRODUCT_COLUMNS)):
    raise AssertionError("Модель вернула матрицу score неожиданного размера")
print(f"Матрица score: {raw_model_scores.shape}")

In [ ]:
pipeline_steps = pd.DataFrame(
    [
        {"этап": name, "класс": type(step).__name__}
        for name, step in candidate.pipeline_.steps
    ]
)
preprocessor = candidate.pipeline_.named_steps["preprocessing"]
preprocessing_steps = pd.DataFrame(
    [
        {
            "ветка": name,
            "pipeline": " → ".join(step_name for step_name, _ in transformer.steps),
            "число входных признаков": len(columns),
        }
        for name, transformer, columns in preprocessor.transformers_
        if hasattr(transformer, "steps")
    ]
)
engineered_preview = engineer_features(x_train.head(3))
display(pipeline_steps)
display(preprocessing_steps)
print(
    f"После feature engineering: {engineered_preview.shape[1]} признаков; "
    f"классификаторов/меток: {len(PRODUCT_COLUMNS)}"
)

## 5. Сравнение baseline и blends

Разреженные метки делают вероятности линейной модели нестабильными для редких продуктов. Поэтому сравниваются сырые score и смеси с popularity prior:

$$score=(1-w)\,score_{SGD}+w\,score_{popularity},\quad w\in\{0,0.25,0.5,0.75\}.$$

Вес выбирается только по MAP@7 апрель→май. Остальные метрики нужны для диагностики: recall показывает долю найденных покупок, precision — полезность семи позиций, coverage — использование каталога, macro PR-AUC — разделимость редких меток вне top-k.

In [ ]:
BLEND_WEIGHTS = [0.0, 0.25, 0.5, 0.75]
for weight in BLEND_WEIGHTS:
    blended_scores = (1.0 - weight) * raw_model_scores + weight * baseline_scores
    comparison[f"sgd_prior_{weight:.2f}"] = evaluate_ranking(
        y_valid, blended_scores, owned=x_valid, k=TOP_K
    )

metric_columns = [
    "map_at_7",
    "recall_at_7",
    "precision_at_7",
    "catalog_coverage_at_7",
    "macro_pr_auc",
    "customers_with_purchase_rate",
]
comparison_frame = (
    pd.DataFrame(comparison).T[metric_columns]
    .sort_values("map_at_7", ascending=False)
)
learned_names = [name for name in comparison if name.startswith("sgd_prior_")]
best_blend_name = max(learned_names, key=lambda name: comparison[name]["map_at_7"])
best_blend_weight = float(best_blend_name.rsplit("_", 1)[1])
best_blend_scores = (
    (1.0 - best_blend_weight) * raw_model_scores
    + best_blend_weight * baseline_scores
)

display(comparison_frame)
print(f"Лучший обучаемый вариант по MAP@7: {best_blend_name}")

In [ ]:
plot_metrics = (
    comparison_frame[["map_at_7", "recall_at_7", "precision_at_7"]]
    .reset_index(names="experiment")
    .melt(id_vars="experiment", var_name="metric", value_name="value")
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_metrics, x="experiment", y="value", hue="metric")
plt.title("Качество baseline и смесей на holdout апрель→май")
plt.xlabel("Эксперимент")
plt.ylabel("Значение метрики")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Метрика")
plt.tight_layout()
plt.show()

## 6. Инвариант: не рекомендовать уже имеющиеся продукты

Даже высокий score не должен возвращать продукт, которым клиент уже владеет в feature-месяце. Следующая ячейка проверяет это для всего holdout и отдельно показывает один компактный пример. Также проверяется, что target, построенный как `0→1`, не пересекается с текущим портфелем.

In [ ]:
recommendations = ranked_recommendations(best_blend_scores, x_valid, k=TOP_K)
owned_matrix = x_valid[PRODUCT_COLUMNS].to_numpy(dtype=bool)
target_matrix = y_valid[PRODUCT_COLUMNS].to_numpy(dtype=bool)
if np.logical_and(owned_matrix, target_matrix).any():
    raise AssertionError("Target 0→1 пересекается с текущими владениями")

violations = []
for row_number, row_recommendations in enumerate(recommendations):
    owned_products = {
        product for product in PRODUCT_COLUMNS if owned_matrix[row_number, PRODUCT_COLUMNS.index(product)]
    }
    recommended_products = [item["product"] for item in row_recommendations]
    overlap = owned_products.intersection(recommended_products)
    if overlap or len(recommended_products) != len(set(recommended_products)):
        violations.append((row_number, sorted(overlap)))

if violations:
    raise AssertionError(f"Найдены нарушения фильтрации: {violations[:3]}")

example_row = 0
example_owned = [
    PRODUCT_DESCRIPTIONS_RU[p]
    for p in PRODUCT_COLUMNS
    if bool(x_valid.iloc[example_row][p])
]
example_recommended = [
    PRODUCT_DESCRIPTIONS_RU[item["product"]]
    for item in recommendations[example_row]
]
example = pd.Series(
    {
        "ncodpers": int(x_valid.iloc[example_row][ID_COLUMN]),
        "уже есть": example_owned,
        "top-7 рекомендаций": example_recommended,
    },
)
print(f"Проверено строк holdout: {len(recommendations)}; нарушений: 0")
display(example.to_frame(name="значение"))

## 7. Чтение сохранённых отчётов production-обучения

Эта секция только читает `reports/model_comparison.json` и `reports/model_metadata.json`; ноутбук ничего не пишет и существующий `ml_models/model.joblib` не перезаписывает. Сохранённые сейчас отчёты и артефакт относятся к прежнему 1%-му прогону. После перехода на полный набор их нужно пересоздать командой `bash scripts/train.sh`, чтобы получить full-data результаты.

In [ ]:
REPORT_DIR = PROJECT_ROOT / "reports"
comparison_path = REPORT_DIR / "model_comparison.json"
metadata_path = REPORT_DIR / "model_metadata.json"
artifact_path = PROJECT_ROOT / "ml_models" / "model.joblib"

if comparison_path.is_file():
    saved_comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    saved_comparison_frame = pd.DataFrame(saved_comparison).T
    available_metrics = [column for column in metric_columns if column in saved_comparison_frame]
    print(f"Прочитан отчёт: {comparison_path.relative_to(PROJECT_ROOT)}")
    display(saved_comparison_frame[available_metrics].sort_values("map_at_7", ascending=False))
else:
    print("reports/model_comparison.json пока отсутствует — сначала запустите полный CLI.")

if metadata_path.is_file():
    saved_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    metadata_keys = [
        "model_version",
        "created_at_utc",
        "target_definition",
        "validation_period",
        "data_scope",
        "loading_mode",
        "random_seed",
        "selected_experiment",
        "selected_prior_weight",
        "artifact_sha256",
    ]
    compact_metadata = {key: saved_metadata.get(key) for key in metadata_keys}
    print(f"Прочитана карточка: {metadata_path.relative_to(PROJECT_ROOT)}")
    display(pd.Series(compact_metadata, name="значение").to_frame())
else:
    print("reports/model_metadata.json пока отсутствует — сначала запустите полный CLI.")

print(f"Production-артефакт существует: {artifact_path.is_file()} (ноутбук его не изменял)")

## 8. Полное обучение и экспорт модели

Финальный refit запускается из корня проекта отдельным процессом. Команда всегда загружает весь CSV и обрабатывает всех клиентов:

```bash
DATA_PATH=train_ver2.csv bash scripts/train.sh   --validation-period 2016-04   --max-iter 60   --seed 42   --artifact ml_models/model.joblib   --report-dir reports
```

Чтобы одновременно логировать параметры, метрики, модель, сигнатуру и артефакты в MLflow, перед запуском задаётся `MLFLOW_TRACKING_URI`, например `http://127.0.0.1:5000`. Полная загрузка и обучение требуют достаточного объёма RAM и могут выполняться существенно дольше прежнего демонстрационного прогона.

CLI сравнит popularity и четыре веса prior, выберет обучаемый вариант по MAP@7, затем переобучит его на train+holdout и атомарно сохранит модель, `model_comparison.json`, `model_metadata.json` и контрольную SHA-256.

## 9. Вывод

- Baseline и обучаемые blends сравниваются на одном временном holdout апрель→май, поэтому нельзя случайно смешать прошлое и будущее.
- MAP@7 остаётся основной метрикой ранжирования; Recall@7, Precision@7, coverage и macro PR-AUC не дают скрыть слабое качество отдельных продуктов.
- Весь preprocessing собран в сериализуемый pipeline и обучается только на train. `seed=42` обеспечивает воспроизводимость модели.
- Фильтрация уже имеющихся продуктов проверяется как отдельный инвариант, а target не включает снятие продукта или сохранение текущего владения.
- В текущем режиме все клиенты обрабатываются вместе: CSV загружается одним вызовом без chunking и sampling. Для выполнения нужен достаточный объём RAM.
- Сохранённые отчёты и артефакт пока отражают прежний 1%-й прогон; full-data результаты создаются командой `bash scripts/train.sh`.